# Lawnmower agent training on Kaggle

Trains the PPO agent of [CL-BZH/Lawnmower](https://github.com/CL-BZH/Lawnmower) on a GPU and compares it with the planners (greedy, boustrophedon, STC) on the example maps.

**Before running:** in the notebook settings set *Accelerator → GPU T4 x2* (or P100) and turn *Internet* on (needed to clone the repository).

A session lasts up to 12 hours. For long trainings use *Save Version → Save & Run All (Commit)*; the files in `/kaggle/working` become the version's output. To continue a run, add that output as an input (*Add Input*) and set `RESUME_FROM` to the checkpoint under `/kaggle/input/...`.

## 1. Setup

In [ ]:
import os
REPO_DIR = "/kaggle/working/Lawnmower"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/CL-BZH/Lawnmower.git {REPO_DIR}
assert os.path.isdir(REPO_DIR), "Cloning failed: check that Internet is ON in the notebook settings."
%cd {REPO_DIR}
!pip install -q -e ".[rl]"
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: training will run on CPU"
import torch; print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 2. Configuration

In [ ]:
# ---- Training configuration ----------------------------------------------
RUN_DIR     = "/kaggle/working/runs/general"   # checkpoints + training_log.csv
MAPS        = ["maps/maze0.txt", "maps/maze1.txt", "maps/maze6.txt",
               "maps/maze8.txt", "maps/test_map.txt"]   # example maps mixed into training
                                               # (maze2 is kept out as a test map)
P_RANDOM    = 0.7            # share of episodes on random gardens
SIZE        = (8, 32)        # outer size range of the random gardens
CURRICULUM  = 0.3            # gardens grow from SIZE[0] to SIZE[1] over this fraction of training (0 = off)
SYMMETRIES  = True           # rotate/mirror every lawn at random (8 symmetries of the square)
STEPS       = 20_000_000     # PPO environment steps
ENVS        = 64             # parallel environments
CHANNELS    = 48             # width of the network
BC_TEACHER  = "greedy"       # planner imitated before PPO ("greedy", "boustrophedon", "stc", or None)
BC_LAWNS    = 2000
TURN_PENALTY = 0.05          # penalty per 45 degrees turned (a step costs 0.1:
                             # 0.05 makes a 90-degree turn cost one step)
RESUME_FROM = None           # e.g. "/kaggle/input/<dataset>/runs/general/agent_final.pt"
SAVE_EVERY  = 1_000_000

# ---- Per-map fine-tuning (optional, run after the general training) ------
# Set FINETUNE_MAP to train further on one map only, from random starts,
# starting from the general agent (RESUME_FROM, or RUN_DIR/agent_final.pt).
FINETUNE_MAP   = None        # e.g. "maps/maze8.txt"
FINETUNE_STEPS = 3_000_000

## 3. Training

In [ ]:
import shlex
args = ["--out", RUN_DIR, "--size", *map(str, SIZE), "--p-random", str(P_RANDOM),
        "--steps", str(STEPS), "--envs", str(ENVS), "--channels", str(CHANNELS),
        "--turn-penalty", str(TURN_PENALTY), "--save-every", str(SAVE_EVERY),
        "--curriculum", str(CURRICULUM)]
if not SYMMETRIES:
    args += ["--no-symmetries"]
if MAPS:
    args += ["--maps", *MAPS]
if RESUME_FROM:
    args += ["--resume", RESUME_FROM]
elif BC_TEACHER:
    args += ["--bc-teacher", BC_TEACHER, "--bc-lawns", str(BC_LAWNS)]
cmd = "python -m lawnmower.learn train " + " ".join(map(shlex.quote, args))
print(cmd)
!{cmd}

In [ ]:
if FINETUNE_MAP:
    base = RESUME_FROM or f"{RUN_DIR}/agent_final.pt"
    name = os.path.splitext(os.path.basename(FINETUNE_MAP))[0]
    ft_dir = f"/kaggle/working/runs/finetune_{name}"
    !python -m lawnmower.learn train --out {ft_dir} --maps {FINETUNE_MAP} --p-random 0 \
        --resume {base} --steps {FINETUNE_STEPS} --envs {ENVS} --lr 1e-4 \
        --turn-penalty {TURN_PENALTY} --save-every {SAVE_EVERY}

## 4. Training curves

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt

log = pd.read_csv(f"{RUN_DIR}/training_log.csv")
w = 20
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(log["step"], log["coverage_%"].rolling(w).mean()); ax[0].set_title("coverage at episode end (%)")
ax[1].plot(log["step"], log["complete_%"].rolling(w).mean()); ax[1].set_title("episodes fully mowed (%)")
ax[2].plot(log["step"], log["overhead"].rolling(w).mean()); ax[2].set_title("overhead of complete episodes (1 = optimal)")
for a in ax: a.set_xlabel("environment steps")
plt.tight_layout(); plt.show()

## 5. Evaluation

Steps needed to mow each map from its usual start (fewer is better), for the planners and the agent.
`agent` plays its most likely move; `agent_best_of_N` samples N episodes and keeps the shortest
(the planners also keep the best of several candidates); `mcts_64` chooses every move with a 64-simulation MCTS guided by the network. Each cell shows steps/turns. maze2 was not used in training.

In [ ]:
CHECKPOINT = f"{RUN_DIR}/agent_final.pt"
!python -m lawnmower.learn eval {CHECKPOINT} maps/maze*.txt maps/test_map.txt --samples 32 --mcts 64
!python -m lawnmower.learn eval {CHECKPOINT} --random 50 --size 8 32 --samples 8

In [ ]:
# Watch the agent on one map (GIF saved in /kaggle/working)
import matplotlib; matplotlib.use("Agg")
from IPython.display import Image
from lawnmower import Lawn
from lawnmower.learn.ppo import load_checkpoint, run_policy
from lawnmower.viz import animate

net, _ = load_checkpoint(CHECKPOINT, torch.device("cuda" if torch.cuda.is_available() else "cpu"))
res = run_policy(net, Lawn.from_file("maps/maze8.txt"), greedy=False, samples=32)
print(res.metrics.as_dict())
animate(res, "/kaggle/working/agent_maze8.gif")
Image("/kaggle/working/agent_maze8.gif")